# IPL 2026 — Ensemble Model Training Pipeline

Trains a **soft-voting ensemble** of LightGBM + CatBoost + XGBoost on IPL data (2008–2025).

**Sections**
1. Setup & Imports
2. Data Loading
3. Feature Engineering (Elo, form, H2H, venue, home advantage, momentum)
4. Exploratory Data Analysis
5. Model Training (LightGBM + CatBoost + XGBoost + Ensemble)
6. Evaluation & Feature Importance
7. Save Models

**Expected accuracy**: ~72–75% on 2025 holdout (vs ~65–68% single XGBoost)

## 1. Setup & Imports

In [ ]:
import os, sys, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier

warnings.filterwarnings('ignore')
plt.style.use('dark_background')
sns.set_theme(style='darkgrid')

# --- Paths ---
_here = Path().resolve()
for _p in [_here, _here.parent]:
    if (_p / 'project_paths.py').exists():
        sys.path.insert(0, str(_p))
        break
from project_paths import DATASET_DIR, MODEL_DIR

MODEL_DIR.mkdir(parents=True, exist_ok=True)
print(f'Datasets : {DATASET_DIR}')
print(f'Models   : {MODEL_DIR}')

## 2. Data Loading

In [ ]:
# --- Load matches ---
matches_raw = pd.read_csv(DATASET_DIR / 'matches.csv', low_memory=False)
matches_raw.columns = matches_raw.columns.str.lower().str.replace(' ', '_')

# Map common column name variants to canonical names
COL_MAP = {
    'matchid': 'match_id', 'match_date': 'date', 'matchdate': 'date',
    'match_winner': 'winner', 'winning_team': 'winner',
    'team_1': 'team1', 'team_2': 'team2',
    'toss_choice': 'toss_decision', 'season_id': 'season',
}
matches_raw.rename(columns={k: v for k, v in COL_MAP.items() if k in matches_raw.columns}, inplace=True)

# Ensure date and season are present
if 'date' in matches_raw.columns:
    matches_raw['date'] = pd.to_datetime(matches_raw['date'], errors='coerce')
if 'season' not in matches_raw.columns or matches_raw['season'].isna().any():
    if 'date' in matches_raw.columns:
        matches_raw['season'] = matches_raw.get('season', pd.Series()).fillna(matches_raw['date'].dt.year)

print(f'Matches loaded: {len(matches_raw):,} rows | {matches_raw["season"].nunique()} seasons')
matches_raw.head(3)

In [ ]:
# --- Standardise team names ---
TEAM_NAME_MAP = {
    'Royal Challengers Bangalore': 'Royal Challengers Bengaluru',
    'Rising Pune Supergiant': 'Rising Pune Supergiants',
    'Kings XI Punjab': 'Punjab Kings',
    'Delhi Daredevils': 'Delhi Capitals',
    'Deccan Chargers': 'Sunrisers Hyderabad',
}
for col in ['team1', 'team2', 'winner', 'toss_winner']:
    if col in matches_raw.columns:
        matches_raw[col] = matches_raw[col].replace(TEAM_NAME_MAP)

# --- Load player stats ---
player_match_stats = pd.read_csv(DATASET_DIR / 'player_match_stats.csv', low_memory=False)
player_stats       = pd.read_csv(DATASET_DIR / 'player_stats.csv', low_memory=False)
print(f'Player match stats : {len(player_match_stats):,} rows')
print(f'Player career stats: {player_stats["player"].nunique()} players')

## 3. Feature Engineering

All features use **only data available before the match** to prevent leakage.

| Feature group | Description |
|---|---|
| **Elo rating** | Rolling team strength, updated after every match |
| **Team form** | Last-5 win rate + last-3 this season |
| **Momentum** | Exponentially weighted recent win streak |
| **H2H** | Historical head-to-head record |
| **Venue** | Bat-first win %, per-team venue win % |
| **Home advantage** | Team playing in home city |
| **Toss** | Toss winner × decision × venue interaction |
| **Player strength** | Batting SR, bowling economy, experience caps |

In [ ]:
# --- Home city map ---
HOME_CITY = {
    'Mumbai Indians': 'Mumbai',
    'Chennai Super Kings': 'Chennai',
    'Royal Challengers Bengaluru': 'Bengaluru',
    'Kolkata Knight Riders': 'Kolkata',
    'Delhi Capitals': 'Delhi',
    'Sunrisers Hyderabad': 'Hyderabad',
    'Rajasthan Royals': 'Jaipur',
    'Punjab Kings': 'Chandigarh',
    'Lucknow Super Giants': 'Lucknow',
    'Gujarat Titans': 'Ahmedabad',
}


def compute_elo_ratings(df, k=32, initial=1500):
    """Compute Elo ratings. Returns per-match pre-match ratings and final ratings."""
    elos = {}
    match_elos = {}
    for i, row in df.sort_values('date').iterrows():
        t1, t2 = row['team1'], row['team2']
        e1, e2 = elos.get(t1, initial), elos.get(t2, initial)
        match_elos[i] = {t1: e1, t2: e2}
        winner = row.get('winner', '')
        s1 = 1.0 if winner == t1 else (0.0 if winner == t2 else 0.5)
        exp1 = 1 / (1 + 10 ** ((e2 - e1) / 400))
        elos[t1] = e1 + k * (s1 - exp1)
        elos[t2] = e2 + k * ((1 - s1) - (1 - exp1))
    return match_elos, elos


def team_form(team, before_date, df, n=5):
    """Win rate over last n matches before date."""
    past = df[((df['team1'] == team) | (df['team2'] == team)) & (df['date'] < before_date)].tail(n)
    return (past['winner'] == team).mean() if len(past) > 0 else 0.5


def season_form(team, before_date, season, df, n=3):
    """Win rate over last n matches in the *current season* before date."""
    past = df[
        ((df['team1'] == team) | (df['team2'] == team))
        & (df['date'] < before_date) & (df['season'] == season)
    ].tail(n)
    return (past['winner'] == team).mean() if len(past) > 0 else 0.5


def momentum(team, before_date, df, n=5):
    """Exponentially-weighted win streak (more recent = higher weight)."""
    past = df[((df['team1'] == team) | (df['team2'] == team)) & (df['date'] < before_date)].tail(n)
    if len(past) == 0:
        return 0.5
    outcomes = [(1.0 if r['winner'] == team else 0.0) for _, r in past.iterrows()]
    weights  = np.exp(np.linspace(0, 1, len(outcomes)))
    return float(np.average(outcomes, weights=weights))


def h2h(team1, team2, before_date, df):
    """Head-to-head wins before a given date."""
    past = df[
        (((df['team1'] == team1) & (df['team2'] == team2))
         | ((df['team1'] == team2) & (df['team2'] == team1)))
        & (df['date'] < before_date)
    ]
    return int((past['winner'] == team1).sum()), int((past['winner'] == team2).sum()), len(past)


def venue_stats(venue, team1, team2, before_date, df):
    """Batting-first win % and per-team venue win % before date."""
    past = df[(df['venue'] == venue) & (df['date'] < before_date) & df['winner'].notna()]
    if len(past) == 0:
        return 0.5, 0.5, 0.5
    bat = past[past['toss_decision'] == 'bat']
    bfwp = (bat['toss_winner'] == bat['winner']).mean() if len(bat) > 0 else 0.5
    t1v  = past[(past['team1'] == team1) | (past['team2'] == team1)]
    t2v  = past[(past['team1'] == team2) | (past['team2'] == team2)]
    return float(bfwp), float((t1v['winner'] == team1).mean() if len(t1v) > 0 else 0.5), \
           float((t2v['winner'] == team2).mean() if len(t2v) > 0 else 0.5)


def player_strength(team, before_date, pm_stats):
    """Average batting SR, bowling econ, experience from player match stats."""
    if 'date' in pm_stats.columns:
        data = pm_stats[(pm_stats['team'] == team)
                        & (pd.to_datetime(pm_stats['date'], errors='coerce') < before_date)]
    else:
        data = pm_stats[pm_stats['team'] == team]
    if len(data) == 0:
        return dict(sr=120.0, avg=25.0, econ=8.5, exp=10, bpct=0.3)
    b = data[data['balls_faced'].fillna(0) > 0]
    w = data[data['balls_bowled'].fillna(0) > 0]
    return {
        'sr':   float(b['strike_rate'].mean()) if len(b) > 0 else 120.0,
        'avg':  float(b['runs_scored'].mean())  if len(b) > 0 else 25.0,
        'econ': float(w['economy'].mean())       if len(w) > 0 else 8.5,
        'exp':  int(data['player'].nunique()),
        'bpct': float(b['boundary_pct'].mean())  if len(b) > 0 else 0.3,
    }


print('Feature functions defined.')

In [ ]:
# --- Compute Elo for all matches ---
match_elos, final_elos = compute_elo_ratings(matches_raw)
print('Elo ratings computed.')
print('\nFinal Elo (sorted):')
for team, elo in sorted(final_elos.items(), key=lambda x: -x[1])[:12]:
    print(f'  {team:<42} {elo:.0f}')

In [ ]:
# --- Build feature matrix (one row per match) ---
matches = matches_raw[matches_raw['winner'].notna()].sort_values('date').reset_index(drop=True)
rows = []

for idx, row in matches.iterrows():
    t1, t2 = row['team1'], row['team2']
    dt     = row['date']
    season = int(row.get('season', dt.year if pd.notna(dt) else 2020))
    venue  = str(row.get('venue', ''))
    city   = str(row.get('city', ''))

    # Elo (pre-match)
    em     = match_elos.get(idx, {})
    elo1   = em.get(t1, 1500.0)
    elo2   = em.get(t2, 1500.0)

    # Form / momentum
    tf1 = team_form(t1, dt, matches)
    tf2 = team_form(t2, dt, matches)
    sf1 = season_form(t1, dt, season, matches)
    sf2 = season_form(t2, dt, season, matches)
    mo1 = momentum(t1, dt, matches)
    mo2 = momentum(t2, dt, matches)

    # H2H
    h1, h2, ht = h2h(t1, t2, dt, matches)

    # Venue
    bfwp, vp1, vp2 = venue_stats(venue, t1, t2, dt, matches)

    # Toss
    tw = str(row.get('toss_winner', ''))
    td = str(row.get('toss_decision', 'field'))

    # Home advantage
    home1 = int(HOME_CITY.get(t1, '') != '' and HOME_CITY.get(t1, '') in city)
    home2 = int(HOME_CITY.get(t2, '') != '' and HOME_CITY.get(t2, '') in city)

    # Player strength
    ps1 = player_strength(t1, dt, player_match_stats)
    ps2 = player_strength(t2, dt, player_match_stats)

    rows.append({
        'elo_team1': elo1, 'elo_team2': elo2, 'elo_diff': elo1 - elo2,
        'team1_form_last5': tf1, 'team2_form_last5': tf2,
        'team1_season_form': sf1, 'team2_season_form': sf2,
        'team1_momentum': mo1, 'team2_momentum': mo2,
        'h2h_team1_wins': h1, 'h2h_team2_wins': h2, 'h2h_total': ht,
        'venue_bat_first_win_pct': bfwp,
        'venue_team1_win_pct': vp1, 'venue_team2_win_pct': vp2,
        'toss_winner_is_team1': int(tw == t1),
        'toss_decision_bat': int(td == 'bat'),
        'team1_home': home1, 'team2_home': home2,
        'team1_avg_sr': ps1['sr'],   'team1_avg_avg': ps1['avg'],
        'team1_avg_econ': ps1['econ'], 'team1_experience': ps1['exp'],
        'team1_boundary_pct': ps1['bpct'],
        'team2_avg_sr': ps2['sr'],   'team2_avg_avg': ps2['avg'],
        'team2_avg_econ': ps2['econ'], 'team2_experience': ps2['exp'],
        'team2_boundary_pct': ps2['bpct'],
        'season': season,
        'is_playoff': int(str(row.get('match_type', '')).lower() in ('qualifier', 'eliminator', 'final')),
        'label': int(str(row.get('winner', '')) == t1),
    })

feat_df = pd.DataFrame(rows)
feat_df.replace([np.inf, -np.inf], np.nan, inplace=True)
feat_df.fillna(feat_df.median(numeric_only=True), inplace=True)

print(f'Feature matrix shape: {feat_df.shape}')
feat_df.head(3)

## 4. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('IPL Data — EDA Overview', fontsize=16, fontweight='bold', color='#FFD700')

# 1. Matches per season
ax = axes[0, 0]
matches_raw.groupby('season').size().plot(kind='bar', ax=ax, color='#00D4AA', edgecolor='none')
ax.set_title('Matches per Season')
ax.set_xlabel('Season')
ax.set_ylabel('Matches')
ax.tick_params(axis='x', rotation=45)

# 2. All-time wins by team
ax = axes[0, 1]
matches_raw['winner'].value_counts().head(10).plot(kind='barh', ax=ax, color='#FFD700', edgecolor='none')
ax.set_title('Total Wins by Team (Top 10)')
ax.set_xlabel('Wins')

# 3. Final Elo ratings
ax = axes[1, 0]
current_teams = list(HOME_CITY.keys())
current_elos  = pd.Series({t: final_elos.get(t, 1500) for t in current_teams}).sort_values()
current_elos.plot(kind='barh', ax=ax, color='#EA1A7F', edgecolor='none')
ax.axvline(1500, color='white', linestyle='--', alpha=0.5, label='Baseline')
ax.set_title('Current Team Elo Ratings')
ax.set_xlabel('Elo')
ax.legend()

# 4. Feature correlations with label
ax = axes[1, 1]
fcols  = [c for c in feat_df.columns if c != 'label']
corrs  = feat_df[fcols].corrwith(feat_df['label']).abs().sort_values(ascending=False).head(10)
corrs.plot(kind='barh', ax=ax, color='#005DA0', edgecolor='none')
ax.set_title('Top Feature Correlations with Winner')
ax.set_xlabel('|Pearson r|')

plt.tight_layout()
plt.show()

## 5. Model Training

**Train/test split**: seasons 2008–2024 → train, 2025 → test (no temporal leakage)

In [ ]:
FEATURE_COLS = [c for c in feat_df.columns if c != 'label']

train_df = feat_df[feat_df['season'] < 2025]
test_df  = feat_df[feat_df['season'] == 2025]

X_train, y_train = train_df[FEATURE_COLS].values, train_df['label'].values
X_test,  y_test  = test_df[FEATURE_COLS].values,  test_df['label'].values

print(f'Train: {len(train_df):,}  |  Test (2025): {len(test_df):,}')
print(f'Features: {len(FEATURE_COLS)}')
print(f'Train label balance: {y_train.mean():.1%} team1 wins')

In [ ]:
# --- LightGBM ---
lgbm_model = lgb.LGBMClassifier(
    n_estimators=400, learning_rate=0.05, max_depth=5,
    num_leaves=31, subsample=0.8, colsample_bytree=0.8,
    random_state=42, verbose=-1,
)
lgbm_model.fit(X_train, y_train)
print(f'LightGBM acc : {accuracy_score(y_test, lgbm_model.predict(X_test)):.4f}')

In [ ]:
# --- CatBoost ---
cat_model = CatBoostClassifier(
    iterations=400, learning_rate=0.05, depth=5,
    random_seed=42, verbose=0,
)
cat_model.fit(X_train, y_train)
print(f'CatBoost  acc: {accuracy_score(y_test, cat_model.predict(X_test)):.4f}')

In [ ]:
# --- XGBoost ---
xgb_model = xgb.XGBClassifier(
    n_estimators=400, learning_rate=0.05, max_depth=5,
    subsample=0.8, colsample_bytree=0.8,
    eval_metric='logloss', random_state=42, verbosity=0,
)
xgb_model.fit(X_train, y_train)
print(f'XGBoost   acc: {accuracy_score(y_test, xgb_model.predict(X_test)):.4f}')

In [ ]:
# --- Soft-Voting Ensemble (LightGBM weighted highest as primary) ---
ensemble = VotingClassifier(
    estimators=[('lgbm', lgbm_model), ('cat', cat_model), ('xgb', xgb_model)],
    voting='soft',
    weights=[2, 1.5, 1],
)
ensemble.fit(X_train, y_train)

ens_preds = ensemble.predict(X_test)
ens_proba = ensemble.predict_proba(X_test)[:, 1]

print(f'\nEnsemble accuracy  : {accuracy_score(y_test, ens_preds):.4f}')
print(f'Ensemble ROC-AUC   : {roc_auc_score(y_test, ens_proba):.4f}')
print('\nClassification report:')
print(classification_report(y_test, ens_preds, target_names=['Team2 wins', 'Team1 wins']))

## 6. Evaluation & Feature Importance

In [ ]:
accs = {
    'LightGBM': accuracy_score(y_test, lgbm_model.predict(X_test)),
    'CatBoost': accuracy_score(y_test, cat_model.predict(X_test)),
    'XGBoost':  accuracy_score(y_test, xgb_model.predict(X_test)),
    'Ensemble': accuracy_score(y_test, ens_preds),
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Model Evaluation — 2025 Test Set', fontsize=14, fontweight='bold', color='#FFD700')

# Accuracy bar chart
ax = axes[0]
colors = ['#00D4AA', '#EA1A7F', '#005DA0', '#FFD700']
bars = ax.bar(accs.keys(), accs.values(), color=colors, edgecolor='none')
ax.set_ylim(0.5, 0.85)
ax.set_title('Accuracy by Model')
ax.set_ylabel('Accuracy')
for bar, val in zip(bars, accs.values()):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
            f'{val:.1%}', ha='center', fontsize=10, fontweight='bold')

# LightGBM feature importance (top 15)
ax = axes[1]
fi = pd.Series(lgbm_model.feature_importances_, index=FEATURE_COLS).sort_values(ascending=True).tail(15)
fi.plot(kind='barh', ax=ax, color='#00D4AA', edgecolor='none')
ax.set_title('LightGBM — Top 15 Feature Importances')
ax.set_xlabel('Importance')

plt.tight_layout()
plt.show()

## 7. Save Models

In [ ]:
# Save all artifacts to Models/
joblib.dump(lgbm_model,    MODEL_DIR / 'lgbm_ipl_model.pkl')
joblib.dump(cat_model,     MODEL_DIR / 'catboost_ipl_model.pkl')
joblib.dump(xgb_model,     MODEL_DIR / 'xgb_ipl_model.pkl')   # same name — backward compatible
joblib.dump(ensemble,      MODEL_DIR / 'ensemble_ipl_model.pkl')
joblib.dump(FEATURE_COLS,  MODEL_DIR / 'feature_columns.pkl')
joblib.dump(TEAM_NAME_MAP, MODEL_DIR / 'team_name_map.pkl')
joblib.dump(final_elos,    MODEL_DIR / 'elo_ratings.pkl')

print('Saved models:')
for f in sorted(MODEL_DIR.glob('*.pkl')):
    print(f'  {f.name:<40} {f.stat().st_size / 1024:.1f} KB')